In [1]:
import os
import json
import time
import logging
import subprocess
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(dotenv_path="../.env")

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

In [2]:
# Verify API key is set
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
elif os.getenv("OPENAI_API_KEY"):
    logger.info("OPENAI_API_KEY is set")
elif os.getenv("ANTHROPIC_API_KEY"):
    logger.info("ANTHROPIC_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

# Check for Cohere API key (needed for re-ranking)
if os.getenv("COHERE_API_KEY"):
    logger.info("COHERE_API_KEY is set (for re-ranking)")
else:
    logger.warning("COHERE_API_KEY not set - re-ranking section will not work")

[2026-03-01 20:08:22,649] p25209 {21338686.py:3} INFO - GROQ_API_KEY is set
[2026-03-01 20:08:22,650] p25209 {21338686.py:13} INFO - COHERE_API_KEY is set (for re-ranking)


In [3]:
from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
# You can change this to other models like:
# - "groq/llama-3.3-70b-versatile" (larger, FREE)
# - "gpt-4o-mini" (OpenAI, paid)
# - "claude-3-5-haiku-20241022" (Anthropic, paid)

MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

/Users/qing0304/Desktop/GU/DSAN6725/spring-2026-a03-Cynthia2734/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/s7/7ttf9mj16ms92j4k34q3lghr0000gn/T/ipykernel_25209/4098160048.py:11: LangChainDeprecationWarning: The class `ChatLiteLLM` was deprecated in LangChain 0.3.24 and will be removed in 1.0. An updated version of the class exists in the `langchain-litellm package and should be used instead. To use it run `pip install -U `langchain-litellm` and import as `from `langchain_litellm import ChatLiteLLM``.
  llm = ChatLiteLLM(
[2026-03-01 20:08:27,347] p25209 {4098160048.py:15} INFO - Using model: groq/llama-3.1-8b-instant


In [4]:
# Data paths - relative to this notebook in notebooks/
CSV_PATH = "../data/structured/daily_sales.csv"
TEXT_DIR = "../data/unstructured"

# Verify data exists
if Path(CSV_PATH).exists():
    logger.info(f"CSV found: {CSV_PATH}")
else:
    logger.error(f"CSV not found at {CSV_PATH}")

text_files = list(Path(TEXT_DIR).glob("*.txt")) if Path(TEXT_DIR).exists() else []
logger.info(f"Found {len(text_files)} product text files in {TEXT_DIR}")

[2026-03-01 20:08:27,397] p25209 {630753970.py:7} INFO - CSV found: ../data/structured/daily_sales.csv
[2026-03-01 20:08:27,399] p25209 {630753970.py:12} INFO - Found 10 product text files in ../data/unstructured


# Explore Data

In [5]:
import pandas as pd

# Load and preview the CSV
df = pd.read_csv(CSV_PATH)
logger.info(f"CSV shape: {df.shape}")
logger.info(f"Columns: {list(df.columns)}")
logger.info(f"Categories: {df['category'].unique().tolist()}")
logger.info(f"Regions: {df['region'].unique().tolist()}")
logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head(3)

[2026-03-01 20:08:30,005] p25209 {83820961.py:5} INFO - CSV shape: (1000, 8)
[2026-03-01 20:08:30,006] p25209 {83820961.py:6} INFO - Columns: ['date', 'product_id', 'product_name', 'category', 'units_sold', 'unit_price', 'total_revenue', 'region']
[2026-03-01 20:08:30,007] p25209 {83820961.py:7} INFO - Categories: ['Books', 'Electronics', 'Food & Grocery', 'Home & Kitchen', 'Pet Supplies', 'Office Supplies', 'Toys & Games', 'Sports & Outdoors', 'Beauty & Personal Care', 'Clothing']
[2026-03-01 20:08:30,007] p25209 {83820961.py:8} INFO - Regions: ['Central', 'West', 'North', 'South', 'East']
[2026-03-01 20:08:30,008] p25209 {83820961.py:9} INFO - Date range: 2024-10-03 to 2024-12-31


,date,product_id,product_name,category,units_sold,unit_price,total_revenue,region
0,2024-10-03,BOOK003,Mystery Novel Collection,Books,42,24.99,1049.58,Central
1,2024-10-03,ELEC002,USB-C Fast Charger,Electronics,57,24.99,1424.43,Central
2,2024-10-03,BOOK001,Python Programming Guide,Books,39,39.65,1546.35,West


In [6]:
# Preview one product text file
sample_file = sorted(Path(TEXT_DIR).glob("*.txt"))[0]
logger.info(f"Sample product file: {sample_file.name}")
logger.info(sample_file.read_text()[:500] + "...")

[2026-03-01 20:08:30,725] p25209 {2853198757.py:3} INFO - Sample product file: BEAU001_product_page.txt
[2026-03-01 20:08:30,726] p25209 {2853198757.py:4} INFO - ========================================
VITAMIN C SERUM - PRODUCT PAGE

Product: Vitamin C Serum
Brand: GlowLab Skincare
Price: $28.99
SKU: BEAU001
Category: Beauty & Personal Care

PRODUCT DESCRIPTION:
Reveal brighter, more youthful skin with GlowLab's Vitamin C Serum. This powerful
antioxidant formula combines 20% Vitamin C (L-Ascorbic Acid) with Vitamin E and
Hyaluronic Acid to reduce dark spots, fine lines, and sun damage while hydrating
and protecti...


# Query Router

In [7]:
def classify_query(question: str) -> str:
    """
    Route a question to the appropriate data source(s).

    Args:
        question: The user's natural language question

    Returns:
        One of: 'csv', 'text', 'both'
    """
    q = question.lower()

    # Keywords that strongly indicate CSV/analytical questions
    csv_keywords = [
        "revenue", "sales", "units sold", "total", "highest", "lowest",
        "region", "volume", "how many", "how much", "average", "sum",
        "december", "october", "november", "month", "quarter",
        "selling", "sold", "price", "trend"
    ]

    # Keywords that strongly indicate unstructured text questions
    text_keywords = [
        "feature", "review", "customer", "description", "spec",
        "what do", "ease", "cleaning", "quality", "rated", "rating",
        "headphone", "air fryer", "product page", "say about", "opinion",
        "feedback", "recommend", "pros", "cons"
    ]

    has_csv = any(kw in q for kw in csv_keywords)
    has_text = any(kw in q for kw in text_keywords)

    if has_csv and has_text:
        return "both"
    elif has_csv:
        return "csv"
    elif has_text:
        return "text"
    else:
        # Default to both for ambiguous queries
        return "both"


# Test on all 6 questions
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

expected = ["csv", "csv", "text", "text", "both", "both"]

for q, exp in zip(test_questions, expected):
    result = classify_query(q)
    status = "✓" if result == exp else "✗"
    logger.info(f"{status} [{result}] (expected: {exp}) {q[:60]}")

[2026-03-01 20:08:32,092] p25209 {2440284005.py:58} INFO - ✓ [csv] (expected: csv) What was the total revenue for Electronics category in Decem
[2026-03-01 20:08:32,092] p25209 {2440284005.py:58} INFO - ✓ [csv] (expected: csv) Which region had the highest sales volume?
[2026-03-01 20:08:32,093] p25209 {2440284005.py:58} INFO - ✓ [text] (expected: text) What are the key features of the Wireless Bluetooth Headphon
[2026-03-01 20:08:32,093] p25209 {2440284005.py:58} INFO - ✓ [text] (expected: text) What do customers say about the Air Fryer's ease of cleaning
[2026-03-01 20:08:32,093] p25209 {2440284005.py:58} INFO - ✓ [both] (expected: both) Which product has the best customer reviews and how well is 
[2026-03-01 20:08:32,094] p25209 {2440284005.py:58} INFO - ✓ [both] (expected: both) I want a product for fitness that is highly rated and sells 


# CSV Retriever

In [8]:
def retrieve_from_csv(question: str) -> str:
    """
    Retrieve relevant data from the sales CSV based on the question.
    Uses pandas for filtering and aggregation, then returns a
    formatted string summary to pass to the LLM.

    Args:
        question: The user's question

    Returns:
        Formatted string with relevant CSV statistics
    """
    logger.info("Retrieving from CSV...")
    q = question.lower()
    df = pd.read_csv(CSV_PATH, parse_dates=["date"])

    parts = []

    # --- Always include overall summary ---
    parts.append("=== Sales Data Overview ===")
    parts.append(f"Total records: {len(df)}")
    parts.append(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    parts.append(f"Total revenue (all time): ${df['total_revenue'].sum():,.2f}")
    parts.append(f"Total units sold (all time): {df['units_sold'].sum():,}")

    # --- Revenue by category ---
    parts.append("\n=== Revenue by Category ===")
    cat_rev = df.groupby("category")["total_revenue"].sum().sort_values(ascending=False)
    parts.append(cat_rev.to_string())

    # --- Revenue by region ---
    parts.append("\n=== Sales Volume (Units) by Region ===")
    reg_units = df.groupby("region")["units_sold"].sum().sort_values(ascending=False)
    parts.append(reg_units.to_string())

    parts.append("\n=== Revenue by Region ===")
    reg_rev = df.groupby("region")["total_revenue"].sum().sort_values(ascending=False)
    parts.append(reg_rev.to_string())

    # --- If question mentions a specific month, add monthly breakdown ---
    if any(m in q for m in ["december", "dec"]):
        parts.append("\n=== December 2024 Revenue by Category ===")
        dec = df[df["date"].dt.month == 12]
        parts.append(dec.groupby("category")["total_revenue"].sum().sort_values(ascending=False).to_string())

    if any(m in q for m in ["october", "oct"]):
        parts.append("\n=== October 2024 Revenue by Category ===")
        oct_df = df[df["date"].dt.month == 10]
        parts.append(oct_df.groupby("category")["total_revenue"].sum().sort_values(ascending=False).to_string())

    if any(m in q for m in ["november", "nov"]):
        parts.append("\n=== November 2024 Revenue by Category ===")
        nov_df = df[df["date"].dt.month == 11]
        parts.append(nov_df.groupby("category")["total_revenue"].sum().sort_values(ascending=False).to_string())

    # --- Top selling products ---
    parts.append("\n=== Top 10 Products by Units Sold ===")
    top_products = df.groupby("product_name")["units_sold"].sum().sort_values(ascending=False).head(10)
    parts.append(top_products.to_string())

    # --- If question mentions fitness or a specific region ---
    if "west" in q:
        parts.append("\n=== West Region - Top Products by Units Sold ===")
        west = df[df["region"] == "West"]
        parts.append(west.groupby("product_name")["units_sold"].sum().sort_values(ascending=False).head(10).to_string())

    if any(kw in q for kw in ["fitness", "sport", "exercise"]):
        parts.append("\n=== Sports Category Sales ===")
        sports = df[df["category"].str.lower().str.contains("sport")]
        if len(sports) > 0:
            parts.append(sports.groupby("product_name")[["units_sold", "total_revenue"]].sum().sort_values("units_sold", ascending=False).to_string())

    context = "\n".join(parts)
    logger.info(f"CSV context length: {len(context)} chars")
    return context


# Quick test
test_csv = retrieve_from_csv("Which region had the highest sales volume?")
logger.info(f"CSV retrieval test (first 500 chars):\n{test_csv[:500]}")

[2026-03-01 20:08:33,664] p25209 {1156651390.py:13} INFO - Retrieving from CSV...
[2026-03-01 20:08:33,672] p25209 {1156651390.py:74} INFO - CSV context length: 1256 chars
[2026-03-01 20:08:33,673] p25209 {1156651390.py:80} INFO - CSV retrieval test (first 500 chars):
=== Sales Data Overview ===
Total records: 1000
Date range: 2024-10-03 to 2024-12-31
Total revenue (all time): $1,738,661.87
Total units sold (all time): 28,804

=== Revenue by Category ===
category
Electronics               413849.85
Clothing                  352019.03
Office Supplies           185907.72
Home & Kitchen            176182.67
Sports & Outdoors         175596.91
Pet Supplies              120314.34
Beauty & Personal Care     94322.77
Books                      89979.62
Toys & Games 


# Text Retriever

In [9]:
def retrieve_from_text(question: str, max_chars: int = 6000) -> str:
    """
    Retrieve relevant content from product text files.
    Scores each file by keyword overlap with the question,
    then returns the top matching file(s).

    Args:
        question:  The user's question
        max_chars: Max characters to return (to avoid token overflow)

    Returns:
        Concatenated content from the most relevant product files
    """
    logger.info("Retrieving from text files...")
    q_words = set(question.lower().split())

    # Score each product file by word overlap with the question
    scores = []
    for txt_file in sorted(Path(TEXT_DIR).glob("*.txt")):
        content = txt_file.read_text(encoding="utf-8")
        content_words = set(content.lower().split())
        overlap = len(q_words & content_words)
        scores.append((overlap, txt_file, content))

    # Sort by score descending
    scores.sort(key=lambda x: x[0], reverse=True)

    parts = []
    total_chars = 0

    # Include top 2 most relevant files (or more if they fit)
    for score, txt_file, content in scores[:3]:
        header = f"=== {txt_file.name} (relevance score: {score}) ==="
        chunk = header + "\n" + content
        if total_chars + len(chunk) > max_chars:
            # Truncate the last file if it would exceed limit
            remaining = max_chars - total_chars
            if remaining > 200:
                parts.append(chunk[:remaining] + "\n... [truncated]")
            break
        parts.append(chunk)
        total_chars += len(chunk)

    context = "\n\n".join(parts)
    logger.info(f"Text context length: {len(context)} chars from {len(scores)} files")
    return context


# Quick test
test_text = retrieve_from_text("What are the key features of the Wireless Bluetooth Headphones?")
logger.info(f"Text retrieval test (first 500 chars):\n{test_text[:500]}")

[2026-03-01 20:08:35,186] p25209 {3496585853.py:14} INFO - Retrieving from text files...
[2026-03-01 20:08:35,189] p25209 {3496585853.py:45} INFO - Text context length: 6020 chars from 10 files
[2026-03-01 20:08:35,190] p25209 {3496585853.py:51} INFO - Text retrieval test (first 500 chars):
=== BEAU001_product_page.txt (relevance score: 5) ===
VITAMIN C SERUM - PRODUCT PAGE

Product: Vitamin C Serum
Brand: GlowLab Skincare
Price: $28.99
SKU: BEAU001
Category: Beauty & Personal Care

PRODUCT DESCRIPTION:
Reveal brighter, more youthful skin with GlowLab's Vitamin C Serum. This powerful
antioxidant formula combines 20% Vitamin C (L-Ascorbic Acid) with Vitamin E and
Hyaluronic Acid to reduce dark spots, f


# Multi-Source Retriever

In [10]:
def retrieve_context(question: str, query_type: str) -> str:
    """
    Route the question to the appropriate data source(s) and
    return the combined context.

    Args:
        question:   The user's question
        query_type: One of 'csv', 'text', 'both'

    Returns:
        Context string to pass to the LLM
    """
    if query_type == "csv":
        return retrieve_from_csv(question)

    elif query_type == "text":
        return retrieve_from_text(question)

    else:  # both
        csv_context = retrieve_from_csv(question)
        text_context = retrieve_from_text(question, max_chars=3000)

        combined = (
            "## STRUCTURED DATA (Sales CSV)\n"
            + csv_context
            + "\n\n## UNSTRUCTURED DATA (Product Pages)\n"
            + text_context
        )
        logger.info(f"Combined context length: {len(combined)} chars")
        return combined


logger.info("retrieve_context() defined")

[2026-03-01 20:08:36,926] p25209 {2580119341.py:33} INFO - retrieve_context() defined


# Retrieval and Generation

In [11]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful data analyst with access to e-commerce sales data and product information. "
    "You are given context retrieved from two possible sources:\n"
    "1. Structured sales CSV data (revenue, units sold, categories, regions, dates)\n"
    "2. Unstructured product pages (descriptions, specifications, customer reviews)\n\n"
    "Answer the user's question based ONLY on the provided context. "
    "Cite specific numbers, product names, or reviews from the context when relevant. "
    "If the context doesn't contain enough information to answer fully, say so clearly. "
    "Keep your answer concise and well-structured.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

logger.info("Prompt template configured")

[2026-03-01 20:08:39,070] p25209 {2968399358.py:22} INFO - Prompt template configured


In [12]:
def answer_question(question: str) -> dict:
    """
    Full pipeline: classify query → retrieve context → generate answer.
    Includes automatic retry on Groq rate limit errors.

    Args:
        question: The user's question

    Returns:
        dict with keys: 'question', 'query_type', 'context', 'answer'
    """
    logger.info(f"Question: {question}")

    # Step 1: Classify query → select data source(s)
    query_type = classify_query(question)
    logger.info(f"Query type: {query_type}")

    # Step 2: Retrieve relevant context
    context = retrieve_context(question, query_type)

    # Step 3: Generate answer with LLM (with retry on rate limit)
    messages = prompt.format_messages(context=context, input=question)

    for attempt in range(5):
        try:
            response = llm.invoke(messages)
            answer = response.content
            logger.info(f"Answer (preview): {answer[:300]}")
            return {
                "question": question,
                "query_type": query_type,
                "context": context,
                "answer": answer,
            }
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 30 * (attempt + 1)
                logger.warning(f"Rate limit hit, waiting {wait}s... (attempt {attempt+1}/5)")
                time.sleep(wait)
            else:
                raise e

    raise RuntimeError("Failed after 5 retries due to rate limiting")


logger.info("answer_question() defined")

[2026-03-01 20:08:42,612] p25209 {1930178464.py:46} INFO - answer_question() defined


In [13]:
# Quick smoke test
test_result = answer_question("Which region had the highest sales volume?")
logger.info(f"Smoke test answer:\n{test_result['answer']}")

[2026-03-01 20:08:43,786] p25209 {1930178464.py:12} INFO - Question: Which region had the highest sales volume?
[2026-03-01 20:08:43,786] p25209 {1930178464.py:16} INFO - Query type: csv
[2026-03-01 20:08:43,787] p25209 {1156651390.py:13} INFO - Retrieving from CSV...
[2026-03-01 20:08:43,794] p25209 {1156651390.py:74} INFO - CSV context length: 1256 chars
20:08:43 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-01 20:08:43,809] p25209 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:08:44 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-01 20:08:44,292] p25209 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-01 20:08:44,294] p25209 {1930178464.py:28} INFO - Answer (preview): Based on the provided context, the region with the highest sales volume is the Central region, with 6,779 units sold.
[2026-03-01 20:08:44

# Answer Test Questions

In [14]:
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

all_results = []

for i, question in enumerate(test_questions, 1):
    logger.info(f"{'='*60}")
    logger.info(f"Question {i}/{len(test_questions)}: {question}")
    result = answer_question(question)
    all_results.append(result)
    logger.info(f"Answer {i}:\n{result['answer']}")

    # Wait between questions to avoid Groq rate limits
    if i < len(test_questions):
        logger.info("Waiting 30s before next question...")
        time.sleep(30)

[2026-03-01 20:08:47,296] p25209 {1920025281.py:13} INFO - ============================================================
[2026-03-01 20:08:47,297] p25209 {1920025281.py:14} INFO - Question 1/6: What was the total revenue for Electronics category in December 2024?
[2026-03-01 20:08:47,298] p25209 {1930178464.py:12} INFO - Question: What was the total revenue for Electronics category in December 2024?
[2026-03-01 20:08:47,298] p25209 {1930178464.py:16} INFO - Query type: csv
[2026-03-01 20:08:47,299] p25209 {1156651390.py:13} INFO - Retrieving from CSV...
[2026-03-01 20:08:47,308] p25209 {1156651390.py:74} INFO - CSV context length: 1668 chars
20:08:47 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-01 20:08:47,310] p25209 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:08:47 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-01 20:08:47,534] p252

# Save Results

In [15]:
# Save all results to part2_results.txt
with open("part2_results.txt", "w") as f:
    f.write("Part 2: Multi-Source RAG with Query Routing - Results\n")
    f.write("=" * 60 + "\n\n")

    for i, result in enumerate(all_results, 1):
        f.write(f"Question {i} [{result['query_type']}]:\n")
        f.write(f"{result['question']}\n")
        f.write("-" * 40 + "\n")
        f.write(f"{result['answer']}\n\n")
        f.write("=" * 60 + "\n\n")

logger.info("Saved to part2_results.txt")

[2026-03-01 20:12:01,260] p25209 {3826533321.py:13} INFO - Saved to part2_results.txt
